# Classical vs. Quantum-Hybrid Ulcer Classifier — full pipeline (GPU)

Runs the exact same pipeline as the repo, on a free Colab GPU, so it isn't at the mercy of a laptop going to sleep mid-training.

**Before running:** Runtime -> Change runtime type -> T4 GPU. Then Runtime -> Run all.

At the end this prints the final accuracy/statistics summary and downloads a `results_bundle.zip` (all of `results/`, `figures/`, and the rebuilt `paper/report.html`) straight to your Downloads folder — Claude can read it from there directly, no need to paste anything back manually unless you want to.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone -q https://github.com/shubhisingh1510/AI.git repo
%cd repo
!git log --oneline -3

In [ ]:
# Colab already ships a CUDA-enabled torch that satisfies requirements.txt's torch>=2.1 /
# torchvision>=0.16 -- only install the packages Colab doesn't already have, so we don't
# risk pip silently swapping in a CPU-only torch build.
!pip install -q pennylane pennylane-lightning grad-cam

In [ ]:
import torch
print('torch', torch.__version__, 'cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'GPU not enabled -- set Runtime > Change runtime type > T4 GPU, then Runtime > Restart and run all.'

## 1. Fetch the exact same data (AZH + the Medetec supplement)

In [ ]:
!python scripts/fetch_azh_dataset.py --config configs/config.yaml
!python scripts/fetch_medetec_supplement.py --config configs/config.yaml

In [ ]:
!python src/data_prep.py --config configs/config.yaml

In [ ]:
!python src/dataset_audit.py --config configs/config.yaml

## 2. Classical ResNet-50 baseline (GPU -- this is the step that was taking hours on CPU)

In [ ]:
!python src/classical_baseline.py --config configs/config.yaml

## 3. Quantum-hybrid classifier

This part is CPU-bound regardless of GPU (PennyLane's `lightning.qubit` state-vector simulator runs on CPU), but Colab's CPU won't fall asleep mid-run the way a laptop does.

In [ ]:
!python src/quantum_hybrid.py --config configs/config.yaml

In [ ]:
!python src/evaluate_compare.py --config configs/config.yaml

## 4. Rebuild the report and print the final numbers

In [ ]:
!python paper/build_report.py

In [ ]:
import json

def show(path):
    with open(path) as f:
        d = json.load(f)
    print(f'--- {path} ---')
    print(json.dumps(d, indent=2)[:3000])
    print()

show('results/classical_metrics.json')
show('results/quantum_metrics.json')
show('results/statistical_tests.json')

import pandas as pd
print('--- results/comparison_table.csv ---')
print(pd.read_csv('results/comparison_table.csv').to_string(index=False))

## 5. Bundle everything and download it to your machine

In [ ]:
import os, zipfile
with zipfile.ZipFile('/content/results_bundle.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for folder in ['results', 'figures', 'data/splits', 'data/metadata']:
        for root, _, files in os.walk(folder):
            for fn in files:
                fp = os.path.join(root, fn)
                zf.write(fp, fp)
    zf.write('paper/report.html', 'paper/report.html')

from google.colab import files
files.download('/content/results_bundle.zip')
print('Downloading results_bundle.zip -- it will land in your Downloads folder.')